<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/06_knowledge_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06: Knowledge extraction (crop requirements and nutrition)

**Purpose:** Turn public agronomic and nutrition sources into structured, sourced tables that (1) set limits for and sanity-check the models and (2) supply the nutrition information shown with each recommendation. Every value keeps its source, and the extraction is evaluated against known answers.

**Sources:**
- FAO ECOCROP, full database file as mirrored in the OpenCLIM/ecocrop repository: temperature, rainfall, soil pH, crop cycle and soil classes (texture, drainage, fertility, salinity). Read directly, since it is already structured.
- FAO/INFOODS Food Composition Table for Western Africa (WAFCT 2019): the Excel database (authoritative values) and the PDF condensed table (used to test extraction).

**Method:**
1. EcoCrop: for each of the 16 crops, select the row of the exact species (with a synonym for tomato), check that the minimum, optimal and maximum values are in order, and save the table with its source.
2. Nutrition: map each crop to a raw-commodity food in the WAFCT Excel (cotton and sugarcane have no food entry); the table holds energy, protein, fat, carbohydrate, fibre, iron, zinc, calcium, vitamin A and other components per 100 g edible portion.
3. Extraction test: values are parsed from the PDF (rule-based parser first, a language model only for rows that fail consistency checks) and scored against the Excel. Accuracy, failure cases and cost are reported.
4. Language model responses are cached, so each page is paid for once.

**Results**
- EcoCrop: 16 of 16 crops matched by exact species name (tomato under its former name, Lycopersicon esculentum); temperature, rainfall and pH minimum, optimal and maximum values are in order for every crop.
- Nutrition: 14 crops mapped to a raw-commodity WAFCT entry; cotton and sugarcane have no food entry.
- PDF extraction test: 14 foods, 56 food-by-column-block pages, 560 values, scored against the Excel database.

| Method | Values handled | Correct |
|---|---|---|
| Rule-based parser | 96.6% | 99.4% of those handled (3 mismatches, all alpha-linolenic acid, where the PDF and Excel disagree) |
| Language model (Gemini Flash) on the 19 values the parser declined | 100% | 94.7% (18 of 19; the error is a retinol value placed in a column the Excel leaves empty) |
| Language model on 10 pages the parser also handled (97 values) | 100% | 99.0%, equal to the rules |
| Hybrid (rules first, model for declined rows) | 100% | 99.3% of all 560 values |

- Language-model usage: 13 calls, 15,288 input and 7,813 output tokens (reasoning tokens included).
- Limits: only 14 of 1,028 foods were tested; the model reads plain text, so blank cells make column alignment ambiguous. Nutrition values in the database come from the Excel; the extraction is evaluated, not relied on.
- Knowledge database: normalised SQLite schema (source, crop, food, crop_food, nutrient, food_nutrient, crop_requirement, crop_soil_class, and a view) with validation checks (foreign keys, ordered ranges, energy and proximate consistency).

**Notes:** EcoCrop values are expert-compiled and assume rain-fed cultivation, so crops that depend on irrigation or floodplains (rice, onion, tomato) need separate treatment. To be completed.

In [ ]:
# @title Cell 1: install
!pip -q install google-genai pdfplumber openpyxl

In [ ]:
# @title Cell 2: setup
import os, json, re, time, glob, hashlib
import numpy as np, pandas as pd
from google.colab import drive, userdata
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'
K = f'{P}/raw/knowledge'
for d in ['wafct2019', 'fao29', 'ecocrop', 'llm_cache']:
    os.makedirs(f'{K}/{d}', exist_ok=True)
CROPS = json.load(open(f'{P}/training_config.json'))['crops']
print(CROPS)

In [ ]:
#@title Cell 3: LLM client with cached calls (optional: needs an API key) { display-mode: "both" }
from google import genai
from google.genai import types
import hashlib
import os
import json
import time

MODEL = 'gemini-3.8-flash'     # Updated from gemini-1.5-flash to match active available model IDs

def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

HAVE_LLM, client = False, None
try:
    _key = get_key('GEMINI_API_KEY')
    if _key:
        client, HAVE_LLM = genai.Client(api_key=_key), True
except Exception as e:
    print('Could not create the client:', type(e).__name__)
if not HAVE_LLM:
    print('No usable API key: language-model steps will be skipped.')

def call_llm(system, user, max_tokens=4000, model=None, json_mode=True):
    if not HAVE_LLM:
        raise RuntimeError('No API key available')
    model = model or MODEL
    key = hashlib.sha256((model + system + user).encode()).hexdigest()[:24]
    path = f'{K}/llm_cache/{key}.json'
    if os.path.exists(path):
        return json.load(open(path))['text']
    cfg = types.GenerateContentConfig(system_instruction=system, max_output_tokens=max_tokens, temperature=0.0,
                                      response_mime_type='application/json' if json_mode else 'text/plain')
    for attempt in range(4):
        try:
            resp = client.models.generate_content(model=model, contents=user, config=cfg)
            break
        except Exception:
            if attempt == 3:
                raise
            time.sleep(5 * 2 ** attempt)          # back off on rate-limit errors
    um = resp.usage_metadata
    usage = {'in': getattr(um, 'prompt_token_count', 0) or 0,
             'out': (getattr(um, 'candidates_token_count', 0) or 0) + (getattr(um, 'thoughts_token_count', 0) or 0)}
    json.dump({'text': resp.text or '', 'usage': usage}, open(path, 'w'))
    return resp.text or ''

def usage_total():
    n = t_in = t_out = 0
    if os.path.exists(f'{K}/llm_cache'):
        for f in os.listdir(f'{K}/llm_cache'):
            u = json.load(open(f'{K}/llm_cache/{f}'))['usage']
            n += 1; t_in += u['in']; t_out += u['out']
    return {'calls': n, 'input_tokens': t_in, 'output_tokens': t_out}

if HAVE_LLM:
    print('Flash models available:', [m.name for m in client.models.list() if 'flash' in m.name][:12])
    print(call_llm('Reply with JSON only.', 'Return {"ok": true}')); print(usage_total())

In [ ]:
#@title LLM client with cached calls (optional: needs an API key) { display-mode: "both" }
from google import genai
from google.genai import types
import hashlib
import json
import os
import time

MODELS = ['gemini-3.8-flash']

def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

HAVE_LLM, client = False, None
try:
    _key = get_key('GEMINI_API_KEY')
    if _key:
        client = genai.Client(api_key=_key, http_options=types.HttpOptions(timeout=60_000))   # 60 s per request
        HAVE_LLM = True
except Exception as e:
    print('Could not create the client:', type(e).__name__)
if not HAVE_LLM:
    print('No usable API key: language-model steps will be skipped.')

def call_llm(system, user, max_tokens=8000, json_mode=True, rounds=3):
    if not HAVE_LLM:
        raise RuntimeError('No API key available')
    path = f'{K}/llm_cache/{hashlib.sha256((system + user).encode()).hexdigest()[:24]}.json'
    if os.path.exists(path):
        return json.load(open(path))['text']
    cfg = types.GenerateContentConfig(system_instruction=system, max_output_tokens=max_tokens, temperature=0.0,
                                      response_mime_type='application/json' if json_mode else 'text/plain')
    last = None
    for rnd in range(rounds):
        for model in MODELS:
            try:
                resp = client.models.generate_content(model=model, contents=user, config=cfg)
                text = resp.text or ''
                if not text:
                    raise ValueError('empty reply')
                um = resp.usage_metadata
                usage = {'in': getattr(um, 'prompt_token_count', 0) or 0,
                         'out': (getattr(um, 'candidates_token_count', 0) or 0) + (getattr(um, 'thoughts_token_count', 0) or 0)}
                json.dump({'text': text, 'model': model, 'usage': usage}, open(path, 'w'))
                return text
            except Exception as e:
                last = e
        time.sleep(10 * (rnd + 1))
    raise RuntimeError(f'All models failed: {type(last).__name__}: {str(last)[:200]}')

def usage_total():
    tot = {}
    for f in os.listdir(f'{K}/llm_cache'):
        d = json.load(open(f'{K}/llm_cache/{f}'))
        t = tot.setdefault(d.get('model', 'unknown'), {'calls': 0, 'input_tokens': 0, 'output_tokens': 0})
        t['calls'] += 1; t['input_tokens'] += d['usage']['in']; t['output_tokens'] += d['usage']['out']
    return tot

if HAVE_LLM:
    print(call_llm('Reply with JSON only.', 'Return {"ok": true}')); print(usage_total())

In [ ]:
# @title Cell 4: look inside the WAFCT files
import pdfplumber
for f in sorted(glob.glob(f'{K}/wafct2019/*')):
    print('\n==', os.path.basename(f), f'{os.path.getsize(f) / 1e6:.1f} MB')
    if f.lower().endswith(('.xlsx', '.xls')):
        xl = pd.ExcelFile(f)
        for s in xl.sheet_names:
            d = xl.parse(s, header=None)
            print(f'  sheet {s!r}: {d.shape[0]} rows x {d.shape[1]} cols')
            print(d.iloc[:6, :9].to_string(max_colwidth=22))
    elif f.lower().endswith('.pdf'):
        with pdfplumber.open(f) as pdf:
            print('  pages:', len(pdf.pages))
            for i, pg in enumerate(pdf.pages):
                t = (pg.extract_text() or '')
                if re.search(r'(?i)sorghum|sorgho', t):
                    print(f'  first page mentioning sorghum: {i + 1} | tables detected: {len(pg.find_tables())}')
                    print(t[:900]); break

In [ ]:
# @title Cell 5: look at the FAO salinity paper
import requests
from bs4 import BeautifulSoup
url = 'https://www.fao.org/3/t0234e/T0234E00.htm'
r = requests.get(url, timeout=60)
print('Status:', r.status_code, '| characters:', len(r.text))
soup = BeautifulSoup(r.text, 'lxml')
links = [(a.get_text(' ', strip=True), a.get('href')) for a in soup.find_all('a', href=True)]
hits = [(t, h) for t, h in links if re.search(r'(?i)toleran|salin|table', t)]
for t, h in hits[:40]:
    print(t[:90], '->', h)

In [ ]:
# @title Cell 6: EcoCrop download, load and check crop coverage
import requests
RAW_URL = 'https://raw.githubusercontent.com/OpenCLIM/ecocrop/main'
for name in ['EcoCrop_DB.csv', 'EcoCrop_DB_secondtrim.csv', 'Ecocrop_variable_lookup.csv']:
    r = requests.get(f'{RAW_URL}/{name}', timeout=60)
    print(name, '| status', r.status_code, f'| {len(r.content) / 1e6:.2f} MB')
    if r.status_code == 200:
        open(f'{K}/ecocrop/{name}', 'wb').write(r.content)

db = pd.read_csv(f'{K}/ecocrop/EcoCrop_DB.csv', low_memory=False, encoding='latin1')
tr = pd.read_csv(f'{K}/ecocrop/EcoCrop_DB_secondtrim.csv', low_memory=False, encoding='latin1')
print('\nFull file:', db.shape, '| trimmed:', tr.shape)
print('Columns in the full file:', list(db.columns))
print('pH / salinity columns:', [c for c in db.columns if re.search(r'(?i)^ph|salin|^sal', c)])

lk = f'{K}/ecocrop/Ecocrop_variable_lookup.csv'
if os.path.exists(lk):
    print('\nVariable lookup:'); print(pd.read_csv(lk, encoding='latin1').to_string(max_colwidth=80))

SCI = {'sorg': 'Sorghum bicolor', 'pmil': 'Pennisetum glaucum', 'cowp': 'Vigna unguiculata',
       'grou': 'Arachis hypogaea', 'maiz': 'Zea mays', 'rice': 'Oryza sativa', 'cass': 'Manihot esculenta',
       'cott': 'Gossypium hirsutum', 'yams': 'Dioscorea rotundata', 'sesa': 'Sesamum indicum',
       'onio': 'Allium cepa', 'soyb': 'Glycine max', 'toma': 'Solanum lycopersicum',
       'bean': 'Phaseolus vulgaris', 'swpo': 'Ipomoea batatas', 'sugc': 'Saccharum officinarum'}
PAR = [c for c in ['TOPMN', 'TOPMX', 'TMIN', 'TMAX', 'ROPMN', 'ROPMX', 'RMIN', 'RMAX', 'GMIN', 'GMAX', 'KTMP', 'ALTMX'] if c in db.columns]

def hits(frame, sci):
    return frame[frame['ScientificName'].astype(str).str.contains(sci, case=False, regex=False, na=False)]

print('\nCrop | rows in full | rows in trimmed | first full match')
for code, sci in SCI.items():
    f, t = hits(db, sci), hits(tr, sci)
    first = {c: f.iloc[0][c] for c in PAR} if len(f) else ''
    print(code, '|', len(f), '|', len(t), '|', (f.iloc[0]['ScientificName'] if len(f) else 'NOT FOUND'), first)

In [ ]:
# @title Cell 7: WAFCT Excel: components and candidate foods per crop
xl = f'{K}/wafct2019/WAFCT_2019.xlsx'
raw = pd.read_excel(xl, sheet_name='03 NV_sum_39 (per 100g EP)', header=None)
names = [re.sub(r'\s+', ' ', str(x)).strip() for x in raw.iloc[0]]
tags = [str(x) for x in raw.iloc[2]]
seen, cols = {}, []
for n in names:
    seen[n] = seen.get(n, 0) + 1
    cols.append(n if seen[n] == 1 else f'{n} #{seen[n]}')
nv = raw.iloc[3:].copy(); nv.columns = cols
nv = nv[nv['Code'].astype(str).str.match(r'^\d{2}_\d{3}')].reset_index(drop=True)
print('Foods in sheet 03:', nv.shape[0], '| columns:', nv.shape[1])
print('\nComponents (name | tag):')
for n, t in list(zip(names, tags))[5:]:
    print(f'  {n} | {t}')

kcal = [c for c in nv.columns if 'kcal' in c][0]
PAT = {'sorg': 'sorghum|guinea corn', 'pmil': 'millet', 'cowp': 'cowpea|black-eyed|niébé',
       'grou': 'groundnut|peanut', 'maiz': 'maize', 'rice': r'\brice\b', 'cass': 'cassava|manioc',
       'cott': 'cotton', 'yams': r'\byam\b', 'sesa': 'sesame', 'onio': 'onion', 'soyb': 'soy',
       'toma': 'tomato', 'bean': r'\bbeans?\b', 'swpo': 'sweet potato', 'sugc': 'sugar ?cane'}
print('\nCandidate foods per crop (code | English name | kcal):')
for c, p in PAT.items():
    m = nv[nv['Food name in English'].astype(str).str.contains(p, case=False, regex=True, na=False)]
    print(f'\n{c}: {len(m)} matches')
    for _, r in m.head(10).iterrows():
        print(f"   {r['Code']} | {str(r['Food name in English'])[:70]} | {r[kcal]}")

In [ ]:
# @title Cell 8: WAFCT PDF: find the food-table pages
import pdfplumber
pdf_path = glob.glob(f'{K}/wafct2019/*.pdf')[0]
code_re = re.compile(r'\b\d{2}_\d{3}\b')
rows = []
with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        t = page.extract_text() or ''
        rows.append((i + 1, len(code_re.findall(t)), len(t)))
pg = pd.DataFrame(rows, columns=['page', 'codes', 'chars'])
dense = pg[pg.codes >= 5]
print('Pages with 5 or more food codes:', len(dense),
      '| range:', int(dense.page.min()) if len(dense) else None, '-', int(dense.page.max()) if len(dense) else None)
print('Pages with almost no text:', int((pg.chars < 50).sum()))
if len(dense):
    first = int(dense.page.iloc[0])
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[first - 1]
        print(f'\nSample page {first}: tables detected = {len(page.find_tables())}')
        print((page.extract_text() or '')[:1500])
else:
    print('No food codes found; send me the output so I can find the right pattern.')

In [ ]:
#@title Cell 9: Build crop_requirements from EcoCrop { display-mode: "both" }
ALT = {'toma': ['Lycopersicon esculentum']}
NUM = ['TOPMN', 'TOPMX', 'TMIN', 'TMAX', 'ROPMN', 'ROPMX', 'RMIN', 'RMAX',
       'PHOPMN', 'PHOPMX', 'PHMIN', 'PHMAX', 'GMIN', 'GMAX', 'KTMP', 'ALTMX']
TXT = ['COMNAME', 'TEXT', 'TEXTR', 'FER', 'FERR', 'SAL', 'SALR', 'DRA', 'DRAR', 'DEP', 'DEPR']

def pick(code):
    names = [SCI[code]] + ALT.get(code, [])
    for n in names:
        m = db[db['ScientificName'].astype(str).str.strip().str.lower() == n.lower()]
        if len(m):
            return m.iloc[0], 'exact'
    for n in names:
        m = db[db['ScientificName'].astype(str).str.contains(n, case=False, regex=False, na=False)
               | db['SYNO'].astype(str).str.contains(n, case=False, regex=False, na=False)]
        if len(m):
            return m.iloc[0], f'partial ({len(m)} rows)'
    return None, 'not found'

rows = []
for code in CROPS:
    r, how = pick(code)
    d = {'crop': code, 'match': how}
    if r is not None:
        d.update({'ecocrop_name': r['ScientificName'], 'ecocrop_code': r['EcoPortCode']})
        d.update({c.lower(): r[c] for c in NUM + TXT if c in r.index})
    rows.append(d)
req = pd.DataFrame(rows)
for c in NUM:
    if c.lower() in req.columns:
        req[c.lower()] = pd.to_numeric(req[c.lower()], errors='coerce')

def ordered(r, cols):
    v = [r[c] for c in cols if c in r.index and pd.notna(r[c])]
    return all(a <= b for a, b in zip(v, v[1:]))
req['temp_ordered'] = req.apply(lambda r: ordered(r, ['tmin', 'topmn', 'topmx', 'tmax']), axis=1)
req['rain_ordered'] = req.apply(lambda r: ordered(r, ['rmin', 'ropmn', 'ropmx', 'rmax']), axis=1)
req['ph_ordered'] = req.apply(lambda r: ordered(r, ['phmin', 'phopmn', 'phopmx', 'phmax']), axis=1)

pd.set_option('display.width', 250)
show = ['crop', 'match', 'ecocrop_name', 'tmin', 'topmn', 'topmx', 'tmax', 'rmin', 'ropmn', 'ropmx', 'rmax',
        'phmin', 'phopmn', 'phopmx', 'phmax', 'gmin', 'gmax', 'sal', 'salr',
        'temp_ordered', 'rain_ordered', 'ph_ordered']
print(req[[c for c in show if c in req.columns]].to_string())
req['source'] = 'FAO EcoCrop (database file as mirrored in OpenCLIM/ecocrop)'
req.to_parquet(f'{P}/crop_requirements.parquet', index=False)

In [ ]:
#@title Cell 10: WAFCT: raw-commodity candidates per crop (for the crop-to-food mapping) { display-mode: "both" }
import os
import pandas as pd
import re

# Ensure 'nv' is defined. If not, load it using the logic from Cell 7
if 'nv' not in globals():
    xl_path = f'{K}/wafct2019/WAFCT_2019.xlsx'
    if os.path.exists(xl_path):
        raw = pd.read_excel(xl_path, sheet_name='03 NV_sum_39 (per 100g EP)', header=None)
        names = [re.sub(r'\s+', ' ', str(x)).strip() for x in raw.iloc[0]]
        seen, cols = {}, []
        for n in names:
            seen[n] = seen.get(n, 0) + 1
            cols.append(n if seen[n] == 1 else f'{n} #{seen[n]}')
        nv = raw.iloc[3:].copy()
        nv.columns = cols
        nv = nv[nv['Code'].astype(str).str.match(r'^\d{2}_\d{3}')].reset_index(drop=True)
    else:
        raise FileNotFoundError(f"WAFCT Excel file not found at {xl_path}. Please make sure Google Drive is mounted and Cell 7 has been executed.")

FIND = {'sorg': r'sorghum', 'pmil': r'millet', 'cowp': r'cowpea', 'grou': r'groundnut|peanut',
        'maiz': r'maize', 'rice': r'\brice\b', 'cass': r'cassava', 'yams': r'\byam\b|dioscorea',
        'bean': r'\bbean', 'swpo': r'sweet potato', 'sesa': r'sesame', 'soyb': r'soy',
        'onio': r'onion', 'toma': r'tomato', 'cott': r'cotton', 'sugc': r'sugar'}

COLS = {'kcal': [c for c in nv.columns if 'kcal' in c][0],
        'protein': [c for c in nv.columns if c.startswith('Protein')][0],
        'iron': [c for c in nv.columns if c.startswith('Iron')][0],
        'zinc': [c for c in nv.columns if c.startswith('Zinc')][0]}
nm = 'Food name in English'

for c, p in FIND.items():
    m = nv[nv[nm].astype(str).str.contains(p, case=False, regex=True, na=False)]
    m = m[m[nm].astype(str).str.contains(r'raw|dry|dried|flour', case=False, regex=True, na=False)]
    m = m[~m[nm].astype(str).str.contains(r'\)\*|bambara|locust|yam bean|false sesame', case=False, regex=True, na=False)]
    print(f'\n{c}: {len(m)} candidates')
    for _, r in m.head(14).iterrows():
        print(f"   {r['Code']} | {str(r[nm])[:68]} | kcal {r[COLS['kcal']]} | protein {r[COLS['protein']]} "
              f"| iron {r[COLS['iron']]} | zinc {r[COLS['zinc']]}")

In [ ]:
#@title Cell 11: Build crop_nutrition from the WAFCT Excel { display-mode: "both" }
MAP = {'sorg': '01_039', 'pmil': '01_017', 'cowp': '03_004', 'grou': '06_010', 'maiz': '01_004',
       'rice': '01_037', 'cass': '02_001', 'yams': '02_034', 'sesa': '06_015', 'onio': '04_018',
       'soyb': '03_008', 'toma': '04_021', 'bean': '03_022', 'swpo': '02_022', 'cott': None, 'sugc': None}
UNITS = {'g', 'mg', 'mcg', 'kJ', 'kcal'}

def unit_of(c):
    m = re.search(r'\(([^()]*)\)\s*$', c)
    return m.group(1) if m and m.group(1) in UNITS else ''

def parse_val(x):
    s = str(x).strip()
    if s in ('', 'nan', 'NaN', 'None'):
        return np.nan, 'missing'
    if s.lower() in ('tr', 'trace'):
        return np.nan, 'trace'
    est = s.startswith('[') and s.endswith(']')
    try:
        return float(s.strip('[]').replace(',', '.')), ('estimated' if est else 'value')
    except ValueError:
        return np.nan, f'text:{s}'

comp_cols, comp_tags = cols[5:], [str(t).strip() for t in tags[5:]]
rows = []
for crop, code in MAP.items():
    if code is None:
        rows.append({'crop': crop, 'food_code': None, 'status': 'no food entry'}); continue
    r = nv[nv['Code'] == code]
    if r.empty:
        rows.append({'crop': crop, 'food_code': code, 'status': 'code not found'}); continue
    r = r.iloc[0]
    for c, t in zip(comp_cols, comp_tags):
        v, flag = parse_val(r[c])
        rows.append({'crop': crop, 'food_code': code, 'food_name': r['Food name in English'],
                     'component': c, 'tag': t, 'unit': unit_of(c), 'value': v, 'flag': flag, 'status': 'ok'})
nut = pd.DataFrame(rows)
nut['source'] = 'FAO/INFOODS WAFCT 2019, sheet 03 (per 100 g edible portion)'

ok = nut[nut.status == 'ok']
print('Crops with a mapped food:', ok.crop.nunique(), '| not mapped:', sorted(nut[nut.status != 'ok'].crop.unique()))
print('\nValue flags:'); print(ok.flag.value_counts())
print('\nShare of missing components per crop (%):')
print((ok.groupby('crop').value.apply(lambda s: s.isna().mean() * 100)).round(0).sort_values(ascending=False).head(8))

KEY = ['Energy (kcal)', 'Water', 'Protein', 'Fat', 'Carbohydrate', 'Fibre', 'Calcium', 'Iron', 'Zinc',
       'Vitamin A (expressed in retinol activity', 'Vitamin C', 'Folate, total']
sel = ok[ok.component.apply(lambda c: any(c.startswith(k) for k in KEY))]
wide = sel.pivot_table(index='crop', columns=sel.component.str.replace(r'\s*\(.*', '', regex=True).str[:14],
                       values='value', aggfunc='first')
pd.set_option('display.width', 250)
print('\nKey nutrients per 100 g edible portion:'); print(wide.round(1).to_string())
nut.to_parquet(f'{P}/crop_nutrition_long.parquet', index=False)

### Crop-to-food mapping (WAFCT 2019, per 100 g edible portion)

| Crop | WAFCT code | Entry | Note |
|---|---|---|---|
| Sorghum | 01_039 | Sorghum, whole grains, raw | |
| Pearl millet | 01_017 | Pearl millet, whole grains (with bran), raw | iron value shared across millet entries |
| Cowpea | 03_004 | Cowpea, dry, raw | |
| Groundnut | 06_010 | Groundnut, shelled, dried, raw | |
| Maize | 01_004 | Maize, white, whole kernel, dry, raw | |
| Rice | 01_037 | Rice, white, raw | brown rice is 01_034 |
| Cassava | 02_001 | Cassava, tuber, white flesh, raw | |
| Yams | 02_034 | Yam, tuber, combined cultivars, raw (Nigeria) | vitamin A not reported |
| Sesame | 06_015 | Sesame seed, whole, dried, raw | |
| Onion | 04_018 | Onion, fresh, raw | |
| Soybean | 03_008 | Soya bean, dry, raw | vitamin C not reported |
| Tomato | 04_021 | Tomato, red, ripe, raw | |
| Dry bean | 03_022 | Bean, white, dry, raw | proxy for common dry bean |
| Sweet potato | 02_022 | Sweet potato, pale flesh, raw | the orange-fleshed entry (02_049) differs in vitamin A |
| Cotton, sugarcane | none | no food entry | shown as "not a food staple" |

Values are for the raw or dry product, not as eaten. Components estimated in the source are flagged; missing components are reported as no data.

In [ ]:
#@title Cell 12: Where do the mapped foods appear in the WAFCT PDF, and how do the rows look? { display-mode: "both" }
import pdfplumber

CODES = {k: v for k, v in MAP.items() if v}
pdf_path = glob.glob(f'{K}/wafct2019/*.pdf')[0]
pages_text = {}
with pdfplumber.open(pdf_path) as pdf:
    for i in range(69, 528):
        pages_text[i + 1] = pdf.pages[i].extract_text() or ''
print('Pages read:', len(pages_text))

def tag_line(text):
    for l in text.split('\n'):
        if l.startswith('INFOODS TAGNAMES'):
            return l
    return None

where = {}
for crop, code in CODES.items():
    where[crop] = [p for p, t in pages_text.items() if re.search(rf'(?m)^{code}\b', t)]
    print(f'{crop} {code}: {len(where[crop])} pages {where[crop]}')

for crop in ['sorg', 'cowp', 'toma']:
    code = CODES[crop]
    for p in where[crop][:4]:
        t = pages_text[p]
        print(f'\n--- {crop} {code} | page {p}')
        print('TAGS:', tag_line(t))
        for l in t.split('\n'):
            if re.match(rf'^{code}\b', l):
                print('ROW :', l)

In [ ]:
#@title Cell 13: WAFCT PDF; rule-based parser scored against the Excel { display-mode: "both" }
NUMTOK = re.compile(r'^(\[?\d+(?:\.\d+)?\]?|\[?tr\]?|-)$', re.I)
KJKCAL = re.compile(r'^(\d+)\((\d+)\)$')

def parse_row(line, code, tags):
    """Read the values from the right-hand end of a table row; return None when the alignment is not certain."""
    rest = re.sub(r'(?<![\w.])(\d{1,2}) (\d{3}\(\d+\))', r'\1\2', line.strip()[len(code):].strip())   # energy: '1 460(345)'
    toks = rest.split()
    has_e = any(t.startswith('ENERC(') for t in tags)
    ncol = [t for t in tags if t != 'EDIBLE1']
    n = len(ncol)
    base = lambda ts: [re.sub(r'\(.*', '', t) for t in ts]
    i = len(toks)
    while i > 0 and (NUMTOK.match(toks[i - 1]) or KJKCAL.match(toks[i - 1])):
        i -= 1
    tail = toks[i:]
    edible = None
    if has_e and len(tail) >= 2 and re.fullmatch(r'\d\.\d{2}', tail[0]) and KJKCAL.match(tail[1]):
        edible, tail = tail[0], tail[1:]
    if has_e and not (tail and KJKCAL.match(tail[0])):
        return None
    need = len(tail) - n
    if need < 0:
        return None
    if need > 0:      # a space used as a thousands separator split a value in two ('1 120')
        cand = [j for j in range(1 if has_e else 0, len(tail) - 1)
                if re.fullmatch(r'[1-9]', tail[j]) and re.fullmatch(r'\d{3}', tail[j + 1])]
        if len(cand) != need:
            return None
        for j in reversed(cand):
            tail[j:j + 2] = [tail[j] + tail[j + 1]]
    out = {}
    if has_e:
        m = KJKCAL.match(tail[0])
        out['ENERC_kJ'], out['ENERC_kcal'] = m.group(1), m.group(2)
        out.update(dict(zip(base(ncol[1:]), tail[1:])))
        if edible:
            out['EDIBLE1'] = edible
    else:
        out.update(dict(zip(base(ncol), tail)))
    return out

def keys_of(tags):
    ks = []
    for t in tags:
        if t == 'EDIBLE1': ks.append('EDIBLE1')
        elif t.startswith('ENERC('): ks += ['ENERC_kJ', 'ENERC_kcal']
        else: ks.append(re.sub(r'\(.*', '', t))
    return ks

def pdf_val(s):
    s = s.strip()
    if s == '-': return np.nan, 'missing'
    if s.strip('[]').lower() == 'tr': return np.nan, 'trace'
    return float(s.strip('[]')), ('estimated' if s.startswith('[') else 'value')

def decimals(s):
    s = s.strip('[]')
    return len(s.split('.')[1]) if '.' in s else 0

def compare(pdf_s, excel_raw):
    pv, pf = pdf_val(pdf_s); ev_, ef = parse_val(excel_raw)
    if np.isnan(pv) or np.isnan(ev_):
        return (np.isnan(pv) and np.isnan(ev_)), pf == ef
    return abs(round(ev_, decimals(pdf_s)) - pv) < 1e-9, pf == ef

tag2col = {}
for c, t in zip(comp_cols, comp_tags):
    t = str(t).strip()
    if t in ('nan', 'None', '') and c.startswith('Sodium'): t = 'NA'      # pandas read the tag "NA" as missing
    if t == 'ENERC': t = 'ENERC_kJ' if '(kJ)' in c else 'ENERC_kcal'
    tag2col[t] = c

recs = []
for crop, code in CODES.items():
    exrow = nv[nv['Code'] == code].iloc[0]
    for p in where[crop][:4]:
        text = pages_text[p]; tl = tag_line(text)
        ptags = tl.split()[2:] if tl else []
        lines = [l for l in text.split('\n') if re.match(rf'^{code}\b', l)]
        res = parse_row(lines[0], code, ptags) if (lines and ptags) else None
        ks = keys_of(ptags)
        if res is not None and 'EDIBLE1' in ks and 'EDIBLE1' not in res:
            res['EDIBLE1'] = '-'
        for k in ks:
            if k not in tag2col:
                continue
            if res is None or k not in res:
                recs.append({'crop': crop, 'code': code, 'page': p, 'key': k, 'parsed': False}); continue
            v_ok, f_ok = compare(res[k], exrow[tag2col[k]])
            recs.append({'crop': crop, 'code': code, 'page': p, 'key': k, 'parsed': True,
                         'pdf': res[k], 'excel': exrow[tag2col[k]], 'value_ok': v_ok, 'flag_ok': f_ok})
ev = pd.DataFrame(recs)
done = ev[ev.parsed]
print('Food x block rows:', ev.groupby(['crop', 'page']).ngroups,
      '| fully parsed:', int(ev.groupby(['crop', 'page']).parsed.all().sum()))
print(f'Values compared: {len(done)} | value matches: {done.value_ok.mean():.1%} | flag matches: {done.flag_ok.mean():.1%}')
print('Values not parsed (would go to the language model):', int((~ev.parsed).sum()))
bad = done[~(done.value_ok & done.flag_ok)]
print('\nMismatches:'); print(bad[['crop', 'page', 'key', 'pdf', 'excel']].head(30).to_string())
ev.to_csv(f'{P}/extraction_eval_rules.csv', index=False)

In [ ]:
#@title Cell 14: Language-model extraction of declined rows and a sample, with the comparison { display-mode: "both" }
N_SAMPLE = 10          # pages the parser handled fine, also sent to the model for a like-for-like comparison
MAX_SECONDS = 900      # stop after this long; answers already received are cached
PRICES = {}            # optional: {'model-id': (US$ per million input tokens, US$ per million output tokens)}
SYSTEM = ("You extract one row from a page of a food composition table. Return JSON only, in the form "
          '{"values": {"TAG": "printed value or null"}}. Take the column names from the INFOODS TAGNAMES line: '
          "remove the unit in brackets (WATER(g) becomes WATER) and split ENERC(kJ(kcal)) into ENERC_kJ and "
          "ENERC_kcal. Include EDIBLE1 only if it is in the tag line (null if the cell is empty). Copy each value "
          "exactly as printed, keeping square brackets and 'tr'. A space used as a thousands separator "
          "(for example '1 120') must be returned without the space ('1120'). Use null for an empty cell or '-'. "
          "If the row has fewer values than columns, return null for the columns you cannot place with certainty. "
          "Never calculate, estimate or correct a value.")

def llm_page(code, page):
    raw = call_llm(SYSTEM, f'Food code: {code}\n\nPage text:\n{pages_text[page]}')
    d = json.loads(raw)['values']
    return {k: (None if v in (None, '', '-') else str(v).strip()) for k, v in d.items()}

decl = ev[~ev.parsed][['crop', 'code', 'page']].drop_duplicates().assign(kind='declined')
okp = ev.groupby(['crop', 'code', 'page']).parsed.all()
okp = okp[okp].reset_index()[['crop', 'code', 'page']]
smp = okp.sample(min(N_SAMPLE, len(okp)), random_state=0).assign(kind='sample')
todo = pd.concat([decl, smp], ignore_index=True)
print(len(decl), 'declined pages and', len(smp), 'sample pages to send')

t_start, llm_res = time.time(), {}
for r in todo.itertuples(index=False):
    if time.time() - t_start > MAX_SECONDS:
        print('Time budget reached; re-run the cell to continue (answers are cached).'); break
    t0 = time.time()
    try:
        llm_res[(r.crop, r.page)] = llm_page(r.code, r.page); status = 'ok'
    except Exception as e:
        llm_res[(r.crop, r.page)] = None; status = f'failed ({type(e).__name__})'
    print(f'{r.kind:8} {r.crop:5} page {r.page}: {status}, {time.time() - t0:.0f} s')

lrecs = []
for r in todo.itertuples(index=False):
    if (r.crop, r.page) not in llm_res:
        continue
    res = llm_res[(r.crop, r.page)]
    exrow = nv[nv['Code'] == r.code].iloc[0]
    for k in keys_of(tag_line(pages_text[r.page]).split()[2:]):
        if k not in tag2col:
            continue
        ok = False
        if res is not None and k in res:
            try:
                v_ok, f_ok = compare(res[k] if res[k] is not None else '-', exrow[tag2col[k]])
                ok = bool(v_ok and f_ok)
            except ValueError:
                ok = False
        lrecs.append({'crop': r.crop, 'page': r.page, 'key': k, 'kind': r.kind, 'llm_ok': ok})
L = pd.DataFrame(lrecs)
m = ev.merge(L, on=['crop', 'page', 'key'], how='left')
m['rule_ok'] = m.parsed & m.value_ok.fillna(False).astype(bool) & m.flag_ok.fillna(False).astype(bool)

d, s = m[m.kind == 'declined'], m[m.kind == 'sample']
print(f'\nRule-based parser: handled {ev.parsed.mean():.1%} of all {len(ev)} values; correct on {m[m.parsed].rule_ok.mean():.1%} of those')
if len(d):
    print(f'Language model on the {len(d)} values the parser declined: correct on {d.llm_ok.astype(bool).mean():.1%}')
if len(s):
    print(f'Sample pages ({len(s)} values): rules {s.rule_ok.mean():.1%} | language model {s.llm_ok.astype(bool).mean():.1%}')
hyb = m.rule_ok.where(m.parsed, m.llm_ok.fillna(False).astype(bool))
print(f'Hybrid (rules first, model for declined rows): correct on {hyb.mean():.1%} of all {len(m)} values')

for mod, t in usage_total().items():
    p = PRICES.get(mod)
    cost = f" | est. cost US$ {t['input_tokens'] / 1e6 * p[0] + t['output_tokens'] / 1e6 * p[1]:.3f}" if p else ''
    print(f"{mod}: {t['calls']} calls, {t['input_tokens']} input and {t['output_tokens']} output tokens{cost}")
print('\nValues the model got wrong:')
print(m[m.kind.notna() & (m.llm_ok == False)][['crop', 'page', 'key', 'excel']].head(20).to_string())
m.to_csv(f'{P}/extraction_eval.csv', index=False)

In [ ]:
#@title Cell 15: Build the normalised knowledge tables and load them into SQLite { display-mode: "both" }
import sqlite3
DB_PATH = f'{P}/knowledge.db'
req = pd.read_parquet(f'{P}/crop_requirements.parquet')
nut = pd.read_parquet(f'{P}/crop_nutrition_long.parquet')

DDL = """
PRAGMA foreign_keys = ON;

CREATE TABLE source (
    source_id TEXT PRIMARY KEY, title TEXT NOT NULL, publisher TEXT, year INTEGER,
    url TEXT, licence TEXT, accessed TEXT);

CREATE TABLE crop (
    crop_id TEXT PRIMARY KEY, name_en TEXT, scientific_name TEXT, spam_code TEXT,
    ecocrop_code TEXT, is_food_staple INTEGER);

CREATE TABLE food (
    food_id TEXT PRIMARY KEY, name_en TEXT, name_fr TEXT,
    source_id TEXT REFERENCES source(source_id), basis TEXT DEFAULT 'per 100 g edible portion');

CREATE TABLE crop_food (
    crop_id TEXT REFERENCES crop(crop_id), food_id TEXT REFERENCES food(food_id),
    rationale TEXT, PRIMARY KEY (crop_id, food_id));

CREATE TABLE nutrient (
    nutrient_id TEXT PRIMARY KEY,            -- INFOODS tagname, e.g. 'FE', 'ENERC_kcal'
    name_en TEXT, unit TEXT);

CREATE TABLE food_nutrient (
    food_id TEXT REFERENCES food(food_id), nutrient_id TEXT REFERENCES nutrient(nutrient_id),
    value REAL,
    flag TEXT CHECK (flag IN ('value', 'estimated', 'trace', 'missing')),
    source_id TEXT REFERENCES source(source_id),
    PRIMARY KEY (food_id, nutrient_id));

CREATE TABLE crop_requirement (
    crop_id TEXT REFERENCES crop(crop_id), variable TEXT,   -- 'temperature', 'rainfall', 'ph', 'cycle'
    abs_min REAL, opt_min REAL, opt_max REAL, abs_max REAL, unit TEXT,
    source_id TEXT REFERENCES source(source_id), PRIMARY KEY (crop_id, variable));

CREATE TABLE crop_soil_class (
    crop_id TEXT REFERENCES crop(crop_id), property TEXT,   -- 'salinity', 'texture', 'drainage'
    optimal TEXT, absolute TEXT, source_id TEXT REFERENCES source(source_id),
    PRIMARY KEY (crop_id, property));

CREATE VIEW v_crop_nutrition AS
SELECT c.crop_id, c.name_en AS crop, n.nutrient_id, n.name_en AS nutrient, n.unit, fn.value, fn.flag
FROM crop c JOIN crop_food cf ON cf.crop_id = c.crop_id
JOIN food_nutrient fn ON fn.food_id = cf.food_id
JOIN nutrient n ON n.nutrient_id = fn.nutrient_id
WHERE fn.flag IN ('value', 'estimated');
"""
open(f'{P}/schema.sql', 'w').write(DDL)

NAMES = {'sorg': 'Sorghum', 'pmil': 'Pearl millet', 'cowp': 'Cowpea', 'grou': 'Groundnut', 'maiz': 'Maize',
         'rice': 'Rice', 'cass': 'Cassava', 'cott': 'Cotton', 'yams': 'Yams', 'sesa': 'Sesame', 'onio': 'Onion',
         'soyb': 'Soybean', 'toma': 'Tomato', 'bean': 'Dry bean', 'swpo': 'Sweet potato', 'sugc': 'Sugarcane'}

source = pd.DataFrame([
    {'source_id': 'wafct2019', 'title': 'FAO/INFOODS Food Composition Table for Western Africa (2019)', 'publisher': 'FAO/INFOODS',
     'year': 2019, 'url': 'https://openknowledge.fao.org/handle/20.500.14283/ca7779b',
     'licence': 'CC BY-NC-SA 3.0 IGO', 'accessed': 'YYYY-MM-DD'},
    {'source_id': 'ecocrop', 'title': 'FAO EcoCrop database (EcoCrop_DB.csv, retrieved via OpenCLIM/ecocrop v1.0)', 'publisher': 'FAO',
     'year': None, 'url': 'https://doi.org/10.5281/zenodo.10845193', 'licence': 'CONFIRM', 'accessed': '2026-11-03'}])

crop = pd.DataFrame({'crop_id': CROPS, 'name_en': [NAMES[c] for c in CROPS]})
crop = crop.merge(req[['crop', 'ecocrop_name', 'ecocrop_code']].rename(
    columns={'crop': 'crop_id', 'ecocrop_name': 'scientific_name'}), on='crop_id', how='left')
crop['ecocrop_code'] = crop.ecocrop_code.astype(str)
crop['spam_code'] = crop.crop_id
crop['is_food_staple'] = (~crop.crop_id.isin(['cott', 'sugc'])).astype(int)

okn = nut[(nut.status == 'ok') & (nut.tag != 'EDIBLE1')].copy()
names = pd.read_excel(f'{K}/wafct2019/WAFCT_2019.xlsx', sheet_name='03 NV_sum_39 (per 100g EP)',
                      header=None, usecols=[0, 1, 2]).iloc[3:]
names.columns = ['food_id', 'name_en', 'name_fr']
food = names[names.food_id.isin(okn.food_code.unique())].drop_duplicates('food_id').copy()
food['source_id'], food['basis'] = 'wafct2019', 'per 100 g edible portion'
crop_food = okn[['crop', 'food_code']].drop_duplicates().rename(columns={'crop': 'crop_id', 'food_code': 'food_id'})
crop_food['rationale'] = 'raw commodity entry chosen as the representative food (notebook 06)'

def nkey(c, t):
    t = str(t).strip()
    if t in ('nan', 'None', '') and c.startswith('Sodium'): return 'NA'
    if t == 'ENERC': return 'ENERC_kJ' if '(kJ)' in c else 'ENERC_kcal'
    return t
okn['nutrient_id'] = [nkey(c, t) for c, t in zip(okn.component, okn.tag)]
nutrient = okn[['nutrient_id', 'component', 'unit']].drop_duplicates('nutrient_id').rename(columns={'component': 'name_en'})
nutrient['name_en'] = (nutrient.name_en.str.replace(r'\s*\((?:kJ|kcal|g|mg|mcg)\)\s*$', '', regex=True)
                       .str.replace(r'\s+', ' ', regex=True).str.strip())
fn = okn[['food_code', 'nutrient_id', 'value', 'flag']].rename(columns={'food_code': 'food_id'})
fn['source_id'] = 'wafct2019'
fn = fn[fn.flag.isin(['value', 'estimated', 'trace', 'missing'])].drop_duplicates(['food_id', 'nutrient_id'])

VARS = {'temperature': ('tmin', 'topmn', 'topmx', 'tmax', 'degC'), 'rainfall': ('rmin', 'ropmn', 'ropmx', 'rmax', 'mm'),
        'ph': ('phmin', 'phopmn', 'phopmx', 'phmax', 'pH')}
rows = []
for _, r in req.iterrows():
    for v, (a, b, c_, d_, u) in VARS.items():
        rows.append({'crop_id': r.crop, 'variable': v, 'abs_min': r[a], 'opt_min': r[b], 'opt_max': r[c_],
                     'abs_max': r[d_], 'unit': u, 'source_id': 'ecocrop'})
    rows.append({'crop_id': r.crop, 'variable': 'cycle', 'abs_min': r['gmin'], 'opt_min': None, 'opt_max': None,
                 'abs_max': r['gmax'], 'unit': 'days', 'source_id': 'ecocrop'})
crop_requirement = pd.DataFrame(rows)
PROPS = {'salinity': ('sal', 'salr'), 'texture': ('text', 'textr'), 'drainage': ('dra', 'drar'),
         'fertility': ('fer', 'ferr'), 'depth': ('dep', 'depr')}
crop_soil_class = pd.DataFrame([{'crop_id': r.crop, 'property': p, 'optimal': r[o], 'absolute': r[a], 'source_id': 'ecocrop'}
                                for _, r in req.iterrows() for p, (o, a) in PROPS.items() if o in req.columns])

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
con = sqlite3.connect(DB_PATH)
con.executescript(DDL)
for name, df in [('source', source), ('crop', crop), ('food', food), ('crop_food', crop_food), ('nutrient', nutrient),
                 ('food_nutrient', fn), ('crop_requirement', crop_requirement), ('crop_soil_class', crop_soil_class)]:
    df.to_sql(name, con, if_exists='append', index=False)
con.commit(); con.close()
print({n: len(d) for n, d in [('source', source), ('crop', crop), ('food', food), ('crop_food', crop_food),
      ('nutrient', nutrient), ('food_nutrient', fn), ('crop_requirement', crop_requirement), ('crop_soil_class', crop_soil_class)]})
print('Database written to', DB_PATH)

In [ ]:
#@title Cell 16: Validate the knowledge database { display-mode: "both" }
con = sqlite3.connect(DB_PATH); con.execute('PRAGMA foreign_keys = ON')
q = lambda s: pd.read_sql(s, con)
pd.set_option('display.width', 220)

print('Foreign key violations:', len(con.execute('PRAGMA foreign_key_check').fetchall()))
bad = q("""SELECT crop_id, variable FROM crop_requirement
           WHERE abs_min > opt_min OR opt_min > opt_max OR opt_max > abs_max OR abs_min > abs_max""")
print('Requirement rows with min/optimal/max out of order:', len(bad))

w = q("SELECT food_id, nutrient_id, value FROM food_nutrient").pivot(index='food_id', columns='nutrient_id', values='value')
ratio = (w['ENERC_kJ'] / w['ENERC_kcal'])
print('Energy kJ/kcal outside 4.0-4.4:', list(ratio[(ratio < 4.0) | (ratio > 4.4)].index))
tot = w[['WATER', 'PROTCNT', 'FAT', 'CHOAVLDF', 'FIBTG', 'ASH']].sum(axis=1)
print('Proximates (water+protein+fat+carbohydrate+fibre+ash) outside 97-103 g per 100 g:',
      {k: round(v, 1) for k, v in tot[(tot < 97) | (tot > 103)].items()})

cov = q("""SELECT crop_id, COUNT(*) AS nutrients_with_value FROM v_crop_nutrition GROUP BY crop_id ORDER BY nutrients_with_value""")
print('\nNutrients with a value per crop (lowest first):'); print(cov.head(6).to_string(index=False))

print('\nProtein, iron and zinc per 100 g (raw commodity):')
print(q("""SELECT crop, nutrient_id, value FROM v_crop_nutrition WHERE nutrient_id IN ('PROTCNT', 'FE', 'ZN')""")
      .pivot(index='crop', columns='nutrient_id', values='value').round(1).sort_values('PROTCNT', ascending=False).to_string())

print('\nPearl millet profile (join across tables):')
print(q("""SELECT r.variable, r.abs_min, r.opt_min, r.opt_max, r.abs_max, r.unit
           FROM crop_requirement r WHERE r.crop_id = 'pmil'""").to_string(index=False))
print(q("SELECT property, optimal, absolute FROM crop_soil_class WHERE crop_id = 'pmil'").to_string(index=False))
con.close()